In [1]:
# Install required libraries
!pip install google-generativeai python-dotenv pymupdf sentence-transformers faiss-cpu numpy

In [2]:
import pymupdf                                           # for reading PDFs
from sentence_transformers import SentenceTransformer   # to convert text into numeric vectors
import faiss                                             # for fast similarity search
import numpy as np
import google.generativeai as genai                     # for generating answers
from dotenv import load_dotenv                          # to load the API key from .env
import os

C:\Users\Faisal Khan\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Faisal Khan\AppData\Local\Temp\ipykernel_9484\1088543063.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai                     # for generating answers


In [3]:
def extract_text(pdf_path):
    doc = pymupdf.open(pdf_path)      # open the PDF
    text = ""
    for page in doc:               # combine text from every page
        text += page.get_text()
    return text

text = extract_text("Placement Policy 2027_B.Tech_MCA.pdf")   # put your PDF path/name here
print(len(text), "characters extracted")

8197 characters extracted


In [4]:
# The whole text can't be processed at once, so we split it into smaller chunks
def chunk_text(text, chunk_size=300):
    words = text.split()           # split text into words
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])   # group every 300 words together
        chunks.append(chunk)
    return chunks

chunks = chunk_text(text)
print(len(chunks), "chunks created")

5 chunks created


In [5]:
# Load a model that converts text into vectors (numbers)
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)   # create a vector for each chunk

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4554.46it/s]


In [6]:
# FAISS is a fast tool for finding similar vectors
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)   # simple distance-based search index
index.add(np.array(embeddings))        # add all chunk embeddings to the index

In [7]:
def get_relevant_chunks(question, k=3):
    q_embedding = model.encode([question])                        # create a vector for the question
    distances, indices = index.search(np.array(q_embedding), k)   # find the top-k most similar chunks
    return [chunks[i] for i in indices[0]]                        # return their text

In [8]:
load_dotenv()   # load the .env file

api_key = os.getenv("GEMINI_API_KEY")   # use the exact variable name from your .env file
genai.configure(api_key=api_key)

gen_model = genai.GenerativeModel("gemini-3.6-flash")

In [9]:
def ask_question(question):
    relevant_chunks = get_relevant_chunks(question)
    context = "\n".join(relevant_chunks)

    prompt = f"""Answer the question based only on the context below.

Context:
{context}

Question:
{question}

Answer:"""

    response = gen_model.generate_content(prompt)

    return "Hello Faisal Khan,\n" + response.text

In [10]:
print(ask_question("what is this pdf about"))

Hello Faisal Khan,
Based on the provided context, the PDF is about the **KIET University Placement Policy for the 2027 Batch (B.Tech./MCA)**. 

It outlines the guidelines, rules, and procedures for campus recruitment and internships, including:
* The "One Student–One Offer" policy.
* Criteria for Dream Offers (₹9.00 LPA or above) and Super Dream Offers (₹16 LPA or above).
* Rules regarding early joining, pre-joining modules, and pre-assessment tests (such as IAMNeo).
* Student eligibility, conduct, discipline, and restrictions regarding contact with company delegates.


In [12]:
print(ask_question("What is the “One Student–One Offer” policy followed by KIET?"))

Hello Faisal Khan,
Based on the provided context, under the **“One Student–One Offer”** policy, after being offered a job by any company, a student is allowed to participate further in the placement process **only** for Super Dream / Dream Companies. As soon as an offer or selection is received, the student must stop appearing in further general drives.
